# Session 3 — Your turn: Four ways to draw a boundary

Same four model families as in the session, new data: two noisy rings, one inside the other. Which family draws the right boundary? And what does scaling do to the one that works from distances?

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np, pandas as pd
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X, y = make_circles(n_samples=600, noise=0.18, factor=0.5, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=1)
print(len(X_train), "training points,", len(X_test), "test points")
from aij_check import check, close, summary

420 training points, 180 test points


## Task 1: Four models, one protocol

Fit the four models below on the training points and store each one's **test accuracy** in `results` (name -> accuracy, rounded to 3 decimals).
Then set `best_name` to the model with the highest test accuracy and `line_gap` to the gap between the best model and logistic regression.

In [2]:
models = {"Logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
          "k-nearest neighbours": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
          "Decision tree (depth 5)": DecisionTreeClassifier(max_depth=5, random_state=0),
          "Random forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=0)}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = round(model.score(X_test, y_test), 3)
best_name = max(results, key=results.get)
line_gap = round(results[best_name] - results["Logistic regression"], 3)
print(results, best_name, line_gap)

{'Logistic regression': 0.494, 'k-nearest neighbours': 0.894, 'Decision tree (depth 5)': 0.883, 'Random forest': 0.878} k-nearest neighbours 0.4


In [3]:
check("four results", lambda: len(results) == 4)
check("logistic regression, a straight line, is near chance on rings", lambda: results["Logistic regression"] < 0.6, "A straight line cannot separate an inner ring from an outer one.")
check("the other three are above 0.8", lambda: len(results) == 4 and all(v > 0.8 for k, v in results.items() if k != "Logistic regression"))
check("best_name names a model with the top score", lambda: results[best_name] == max(results.values()))
check("line_gap is the gap to the straight line", lambda: close(line_gap, max(results.values()) - results["Logistic regression"], 0.0011))
summary()

PASS  four results
PASS  logistic regression, a straight line, is near chance on rings
PASS  the other three are above 0.8
PASS  best_name names a model with the top score
PASS  line_gap is the gap to the straight line

5 of 5 checks passed.  Well done.


**Why it works.** Every model has a built-in preference. A straight line cannot enclose a region, so logistic regression stays near 50% here; the neighbour, tree and forest models bend with the data and handle the rings. The best of the three differ by a few points on 180 test points, which is within noise, so the table does not crown a single winner.

## Task 2: When the units are wrong

Make `X_wide` by multiplying the **second** feature by 1000 (as if it were measured in millimetres while the first is in metres), split it with the same settings as above, and measure the test accuracy of k-NN (15 neighbours) first **without** scaling (`raw_accuracy`) and then **with** `StandardScaler` in front (`scaled_accuracy`).

In [4]:
X_wide = X.copy(); X_wide[:, 1] *= 1000
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_wide, y, test_size=0.3, stratify=y, random_state=1)
raw_accuracy = KNeighborsClassifier(n_neighbors=15).fit(Xw_train, yw_train).score(Xw_test, yw_test)
scaled_accuracy = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)).fit(Xw_train, yw_train).score(Xw_test, yw_test)
print(raw_accuracy, scaled_accuracy)

0.7166666666666667 0.8944444444444445


In [5]:
check("the second feature was stretched", lambda: np.abs(X_wide[:, 1]).max() > 500)
check("without scaling k-NN suffers", lambda: raw_accuracy < 0.8, "Distances are now dominated by the second feature.")
check("with scaling it recovers", lambda: scaled_accuracy > 0.85)
check("scaling is worth more than 10 points", lambda: scaled_accuracy - raw_accuracy > 0.10)
summary()

PASS  the second feature was stretched
PASS  without scaling k-NN suffers
PASS  with scaling it recovers
PASS  scaling is worth more than 10 points

4 of 4 checks passed.  Well done.


**Why it works.** k-NN works from distances, so a feature with a large unit decides every distance on its own. Scaling puts the features on a comparable footing, and the pipeline learns the scale from the training points only.